# Predict and save metaphor_features (Train and Test Sets)

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaConfig, RobertaModel
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
from collections import defaultdict
import numpy as np
from tqdm import tqdm
from types import SimpleNamespace
import spacy

# Download NLTK data
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# Load spaCy model (requires pre-downloading 'en_core_web_sm')
try:
    nlp = spacy.load('en_core_web_sm')
    print("spaCy model loaded successfully.")
except OSError:
    print("Error: Please run 'python -m spacy download en_core_web_sm' to download the model.")
    exit(1)

# POS Tag Mapping (copied from Metaphor.txt, 20 tags)
pos_to_id = defaultdict(lambda: len(pos_to_id))
id_to_pos = {}
common_pos_tags = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "ADP", "NUM", "CONJ", "INTJ", "PART", "SCONJ", "SYM", "X", ".", "AUX", "PROPN", "PUNCT", "SPACE", "CCONJ"]
for tag in common_pos_tags:
    pos_to_id[tag]

# Mapping from spaCy to custom POS tags (handling the 18 vs 20 difference)
spacy_to_custom = {
    'ADJ': 'ADJ',
    'ADP': 'ADP',
    'ADV': 'ADV',
    'AUX': 'AUX',
    'CCONJ': 'CCONJ',
    'DET': 'DET',
    'INTJ': 'INTJ',
    'NOUN': 'NOUN',
    'NUM': 'NUM',
    'PART': 'PART',
    'PRON': 'PRON',
    'PROPN': 'PROPN',
    'PUNCT': 'PUNCT',  # Includes '.'
    'SCONJ': 'SCONJ',
    'SYM': 'SYM',
    'VERB': 'VERB',
    'X': 'X',
    'SPACE': 'SPACE',
    # Extra mapping: CONJ -> CCONJ, . -> PUNCT
    'CONJ': 'CCONJ'  # In case older tags occur
}

# Concreteness Scorer Class (copied from Metaphor.txt)
class ConcretenessScorer:
    def __init__(self, file_path):
        print(f"Loading concreteness ratings: {file_path}")
        df = pd.read_excel(file_path)
        word_col = 'Word'
        score_col = 'Conc.M'
        self.concreteness_dict = pd.Series(df[score_col].values, index=df[word_col].str.lower()).to_dict()
        print(f"Loaded {len(self.concreteness_dict)} ratings.")

    def get_score(self, word):
        return self.concreteness_dict.get(str(word).lower(), 3.0)

    def get_wordnet_semantic_features(self, target_word, context_words):
        def get_hypernym_path(synset):
            path = []
            current = synset
            while current:
                path.append(current)
                hypernyms = current.hypernyms()
                current = hypernyms[0] if hypernyms else None
            return path

        features = []
        target_synsets = wn.synsets(target_word)
        if not target_synsets:
            return [0.0, 0.0, 0.0]

        target_synset = target_synsets[0]
        target_path = get_hypernym_path(target_synset)

        path_distances = []
        lch_depths = []
        for context_word in context_words:
            context_synsets = wn.synsets(context_word)
            if not context_synsets:
                continue
            context_synset = context_synsets[0]
            context_path = get_hypernym_path(context_synset)

            try:
                path_distance = target_synset.shortest_path_distance(context_synset) or 10.0
            except:
                path_distance = 10.0
            path_distances.append(path_distance)

            try:
                lch = target_synset.lowest_common_hypernyms(context_synset)
                lch_depth = lch[0].max_depth() if lch else 0.0
            except:
                lch_depth = 0.0
            lch_depths.append(lch_depth)

        avg_path_distance = np.mean(path_distances) if path_distances else 0.0
        avg_lch_depth = np.mean(lch_depths) if lch_depths else 0.0
        semantic_domain_diff = 1.0 if avg_path_distance > 5.0 else 0.0
        return [float(avg_path_distance), float(avg_lch_depth), float(semantic_domain_diff)]

# Metaphor Model Class (copied from Metaphor.txt, used for sentence-level prediction)
class MetaphorRoberta(nn.Module):
    def __init__(self, config, num_labels=2, num_pos_tags=len(pos_to_id), pos_embedding_dim=64, num_concreteness_features=6):
        super(MetaphorRoberta, self).__init__()
        self.num_labels = num_labels
        self.config = config
        self.encoder = RobertaModel(config=config)
        self.args = SimpleNamespace(drop_ratio=0.1, classifier_hidden=128, small_mean=True)
        self.dropout = nn.Dropout(self.args.drop_ratio)
        self.pos_embedding = nn.Embedding(num_embeddings=num_pos_tags, embedding_dim=pos_embedding_dim)
        self.self_attention = nn.MultiheadAttention(embed_dim=self.config.hidden_size, num_heads=8, dropout=self.args.drop_ratio, batch_first=True)

        input_dim_for_target_with_pos = self.config.hidden_size + pos_embedding_dim
        self.SPV_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)
        self.MIP_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)

        gate_input_dim = self.config.hidden_size + input_dim_for_target_with_pos
        self.gate_mlp = nn.Sequential(
            nn.Linear(gate_input_dim, self.args.classifier_hidden),
            nn.ReLU(),
            nn.Dropout(self.args.drop_ratio),
            nn.Linear(self.args.classifier_hidden, 2)
        )

        linguistic_feature_dim = 16
        self.linguistic_mlp = nn.Sequential(
            nn.Linear(num_concreteness_features, linguistic_feature_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(linguistic_feature_dim * 2, linguistic_feature_dim)
        )

        self.classifier = nn.Linear(self.args.classifier_hidden + linguistic_feature_dim, num_labels)
        self.logsoftmax = nn.LogSoftmax(dim=1)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embedding.weight, mean=0.0, std=self.config.initializer_range)
        for module in [self.SPV_linear, self.MIP_linear, self.classifier]:
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        for mlp in [self.gate_mlp, self.linguistic_mlp]:
            for module in mlp:
                if isinstance(module, nn.Linear):
                    module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                    if module.bias is not None:
                        module.bias.data.zero_()

    def forward(self, input_ids, input_ids_2, target_mask, target_mask_2, attention_mask, attention_mask_2, pos_ids=None, concreteness_features=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        attn_output, _ = self.self_attention(sequence_output, sequence_output, sequence_output, key_padding_mask=~attention_mask.bool())
        sequence_output = attn_output
        cls_output = sequence_output[:, 0, :]

        target_output = sequence_output * target_mask.unsqueeze(2)
        target_output = self.dropout(target_output)
        target_output = target_output.sum(dim=1) / (target_mask.sum(-1, keepdim=True) + 1e-10)

        pos_embedding = self.pos_embedding(pos_ids)
        target_output_with_pos = torch.cat([target_output, pos_embedding], dim=1)

        outputs_2 = self.encoder(input_ids=input_ids_2, attention_mask=attention_mask_2)
        sequence_output_2 = outputs_2.last_hidden_state
        target_output_2 = sequence_output_2 * target_mask_2.unsqueeze(2)
        target_output_2 = self.dropout(target_output_2)
        target_output_2 = target_output_2.sum(dim=1) / (target_mask_2.sum(-1, keepdim=True) + 1e-10)

        SPV_hidden = self.SPV_linear(torch.cat([cls_output, target_output_with_pos], dim=1))
        MIP_hidden = self.MIP_linear(torch.cat([target_output_2, target_output_with_pos], dim=1))

        gate_input = torch.cat([cls_output, target_output_with_pos], dim=1)
        gate_weights = torch.sigmoid(self.gate_mlp(gate_input))
        SPV_weight, MIP_weight = gate_weights[:, 0:1], gate_weights[:, 1:2]
        combined_hidden = SPV_weight * SPV_hidden + MIP_weight * MIP_hidden

        linguistic_embedding = self.linguistic_mlp(concreteness_features)
        final_features = torch.cat([combined_hidden, linguistic_embedding], dim=1)

        logits = self.classifier(self.dropout(final_features))
        logits = self.logsoftmax(logits)
        return {'logits': logits}

# Book Review Dataset Processing Class (modified for word-by-word prediction: loops over each w_index to predict metaphor_label)
class BookReviewMetaphorDataset(Dataset):
    def __init__(self, data, tokenizer, concreteness_scorer, model, max_len=128):
        self.sentences = [item['text'] for item in data]
        self.labels = [item['label'] for item in data]
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.original_sentences = [item['text'] for item in data]
        self.concreteness_scorer = concreteness_scorer
        self.model = model
        if self.concreteness_scorer is None:
            raise ValueError("ConcretenessScorer instance must be provided.")
        self.device = next(self.model.parameters()).device

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        label = self.labels[idx]
        original_sentence = self.original_sentences[idx]

        words = original_sentence.split()
        # Generate w_index label list (index of each word 0,1,2,...)
        w_indices_list = list(range(len(words)))
        w_indices_str = str(w_indices_list)

        # Word-by-word prediction: loop through each w_index
        metaphor_labels = []
        all_concreteness_features = []  # Optional: collect concreteness_features for each word

        for w_index in w_indices_list:
            target_word = words[w_index]
            literal_meaning = ""
            synsets = wn.synsets(target_word)
            if synsets:
                literal_meaning = synsets[0].definition()
            # Modify sentence: replace target_word
            modified_words = words.copy()
            modified_words[w_index] = f"[TARGET]{target_word}[/TARGET]"
            modified_sentence = " ".join(modified_words) + f" [LITERAL] {literal_meaning}"

            encoding = self.tokenizer(modified_sentence, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids = encoding['input_ids'].flatten().to(self.device)

            target_mask = torch.zeros(self.max_len, dtype=torch.float, device=self.device)
            target_tokens_str = self.tokenizer.encode(f"[TARGET]{target_word}[/TARGET]", add_special_tokens=False)
            for i in range(len(input_ids) - len(target_tokens_str) + 1):
                if torch.equal(input_ids[i:i+len(target_tokens_str)], torch.tensor(target_tokens_str, device=self.device)):
                    target_mask[i:i+len(target_tokens_str)] = 1.0
                    break

            target_sentence = target_word if target_word else "[PAD]"
            encoding_2 = self.tokenizer(target_sentence, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids_2 = encoding_2['input_ids'].flatten().to(self.device)

            target_mask_2 = torch.zeros(self.max_len, dtype=torch.float, device=self.device)
            target_tokens_2 = self.tokenizer.encode(target_word, add_special_tokens=False)
            for i in range(len(input_ids_2) - len(target_tokens_2) + 1):
                if torch.equal(input_ids_2[i:i+len(target_tokens_2)], torch.tensor(target_tokens_2, device=self.device)):
                    target_mask_2[i:i+len(target_tokens_2)] = 1.0
                    break

            # Calculate concreteness features
            target_concreteness_score = self.concreteness_scorer.get_score(target_word)
            context_words = [word for i, word in enumerate(original_sentence.split()) if i != w_index]
            if context_words:
                context_concreteness_score = sum(self.concreteness_scorer.get_score(w) for w in context_words) / len(context_words)
            else:
                context_concreteness_score = 3.0

            # Calculate WordNet semantic features
            semantic_features = self.concreteness_scorer.get_wordnet_semantic_features(target_word, context_words)
            concreteness_features = torch.tensor([
                target_concreteness_score,
                context_concreteness_score,
                abs(target_concreteness_score - context_concreteness_score)
            ] + semantic_features, dtype=torch.float).to(self.device).unsqueeze(0)

            # POS for target_word (using spaCy)
            target_pos_tag = 'NOUN'  # default
            target_doc = nlp(target_word)
            if target_doc:
                target_pos_tag = spacy_to_custom.get(target_doc[0].pos_, 'NOUN')
            pos_id = torch.tensor(pos_to_id[target_pos_tag], dtype=torch.long).to(self.device).unsqueeze(0)

            # Predict metaphor label for this word
            input_ids_batch = input_ids.unsqueeze(0)
            input_ids_2_batch = input_ids_2.unsqueeze(0)
            target_mask_batch = target_mask.unsqueeze(0)
            target_mask_2_batch = target_mask_2.unsqueeze(0)
            attention_mask = encoding['attention_mask'].flatten().to(self.device).unsqueeze(0)
            attention_mask_2 = encoding_2['attention_mask'].flatten().to(self.device).unsqueeze(0)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids_batch, input_ids_2=input_ids_2_batch,
                                     target_mask=target_mask_batch, target_mask_2=target_mask_2_batch,
                                     attention_mask=attention_mask, attention_mask_2=attention_mask_2,
                                     pos_ids=pos_id, concreteness_features=concreteness_features)
                logits = outputs['logits']
                word_metaphor_label = torch.argmax(logits, dim=1).item()  # 0 or 1
                metaphor_labels.append(word_metaphor_label)

            # Optional: collect concreteness_features (if needed per-word)
            all_concreteness_features.append(concreteness_features.squeeze(0).cpu().tolist())

        # Sentence-level POS tagging (word-level)
        doc = nlp(original_sentence)
        pos_tags = [spacy_to_custom.get(token.pos_, 'NOUN') for token in doc]
        pos_tags = pos_tags[:self.max_len]
        while len(pos_tags) < self.max_len:
            pos_tags.append('PAD')
        pos_str = str(pos_tags)

        # Average concreteness_features (or save as list)
        avg_concreteness = np.mean(all_concreteness_features, axis=0).tolist()
        concreteness_str = str(avg_concreteness)

        return {
            'text': original_sentence,
            'label': label,
            'metaphor_labels': str(metaphor_labels),  # Word-by-word prediction list [0,1,0,...]
            'concreteness_features': concreteness_str,  # Average or per-word list
            'pos_tags': pos_str,
            'w_indices': w_indices_str  # Word index list for each sentence
        }

# Main Function
def main():
    # Paths
    model_path = "/content/drive/MyDrive/PhD/Metaphor_model/Metaphor_model_ConcreteMLP4+Contrast+BEST"
    concreteness_path = "/content/drive/MyDrive/PhD/Metaphor_model/Concreteness ratings.xlsx"
    book_dir = "/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/"
    train_csv_path = os.path.join(book_dir, "train.csv")
    test_csv_path = os.path.join(book_dir, "test.csv")
    save_dir = "/content/drive/MyDrive/PhD/Sentiment_model/Dataset/"
    os.makedirs(save_dir, exist_ok=True)
    output_train_csv = os.path.join(save_dir, "train_book_with_metaphor_pos.csv")
    output_test_csv = os.path.join(save_dir, "test_book_with_metaphor_pos.csv")

    # Load tokenizer and model
    tokenizer = RobertaTokenizer.from_pretrained(model_path)
    tokenizer.add_tokens(['[TARGET]', '[/TARGET]', '[LITERAL]'])
    config = RobertaConfig.from_pretrained(model_path)
    model = MetaphorRoberta(config, num_pos_tags=len(pos_to_id))
    model.load_state_dict(torch.load(os.path.join(model_path, "pytorch_model.bin")))
    model.encoder.resize_token_embeddings(len(tokenizer))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    # Load ConcretenessScorer
    concreteness_scorer = ConcretenessScorer(concreteness_path)

    # Load Book Review CSV
    print("Loading Book Review Dataset...")
    train_df_raw = pd.read_csv(train_csv_path)
    test_df_raw = pd.read_csv(test_csv_path)
    train_data = [{'text': row['text'], 'label': row['label']} for _, row in train_df_raw.iterrows()]
    test_data = [{'text': row['text'], 'label': row['label']} for _, row in test_df_raw.iterrows()]
    print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

    # Predict and annotate train set
    print("Processing Train Set (word-by-word metaphor prediction + concreteness/WordNet features + w_index tags)...")
    train_dataset = BookReviewMetaphorDataset(train_data, tokenizer, concreteness_scorer, model)
    train_df = pd.DataFrame([
        {'review': item['text'], 'label': item['label'],
         'metaphor_labels': item['metaphor_labels'],
         'concreteness_features': item['concreteness_features'],
         'pos_tags': item['pos_tags'],
         'w_indices': item['w_indices']}
        for item in tqdm(train_dataset)
    ])
    train_df.to_csv(output_train_csv, index=False)
    print(f"Train set saved to: {output_train_csv} (Rows: {len(train_df)})")

    # Predict and annotate test set
    print("Processing Test Set (word-by-word metaphor prediction + concreteness/WordNet features + w_index tags)...")
    test_dataset = BookReviewMetaphorDataset(test_data, tokenizer, concreteness_scorer, model)
    test_df = pd.DataFrame([
        {'review': item['text'], 'label': item['label'],
         'metaphor_labels': item['metaphor_labels'],
         'concreteness_features': item['concreteness_features'],
         'pos_tags': item['pos_tags'],
         'w_indices': item['w_indices']}
        for item in tqdm(test_dataset)
    ])
    test_df.to_csv(output_test_csv, index=False)
    print(f"Test set saved to: {output_test_csv} (Rows: {len(test_df)})")

    # Display example output
    print("\nTrain set example row:")
    print(train_df.head(1).to_string(index=False))
    print("\nTest set example row:")
    print(test_df.head(1).to_string(index=False))

    # Verify unique POS tags
    all_pos_train = set()
    for row in train_df['pos_tags']:
        pos_list = eval(row)
        all_pos_train.update(set(p for p in pos_list if p != 'PAD'))
    print(f"\nUnique POS tag count in Train set: {len(all_pos_train)} - {sorted(all_pos_train)}")

if __name__ == "__main__":
    main()

# Sentiment Analysis using metaphor_features

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import ast
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, RobertaForSequenceClassification, RobertaConfig
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ==========================================
# 1. Metaphor Fusion Model without Attention (Concatenation Baseline)
# ==========================================
class RobertaMetaphorNoAttention(RobertaForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        # 1. Metaphor Feature Embedding
        self.metaphor_embedding = nn.Embedding(2, config.hidden_size)
        self.dropout = nn.Dropout(0.1)

        # 2. Projection Layer (Replaces Attention)
        # Input dimension is text_hidden + meta_hidden (i.e. 2 * hidden_size)
        # Output dimension maps back to hidden_size
        self.fusion_projection = nn.Sequential(
            nn.Linear(config.hidden_size * 2, config.hidden_size),
            nn.Tanh(),
            nn.Dropout(0.1)
        )

        # Initialize custom weights
        self._init_custom_weights()

    def _init_custom_weights(self):
        for module in self.fusion_projection:
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                if module.bias is not None:
                    module.bias.data.zero_()

    def forward(self, input_ids, attention_mask=None, metaphor_features=None, labels=None):
        if metaphor_features is None:
            raise ValueError("Model requires 'metaphor_features' input.")

        # Text Features: (Batch, Seq_Len, Hidden)
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]

        # Metaphor Features: (Batch, Seq_Len, Hidden)
        metaphor_embeds = self.metaphor_embedding(metaphor_features)
        metaphor_embeds = self.dropout(metaphor_embeds)

        # --- Core Modification: Direct concatenation and fusion without Attention weights ---
        # 1. Concatenation
        combined_features = torch.cat((sequence_output, metaphor_embeds), dim=-1) # (Batch, Seq, 2*Hidden)

        # 2. Linear Projection Fusion
        fused_output = self.fusion_projection(combined_features) # (Batch, Seq, Hidden)

        # --- Classification Logic ---
        # Simple Mean Pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(fused_output.size()).float()
        sum_embeddings = torch.sum(fused_output * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        # Classifier
        logits = self.classifier(pooled_output.unsqueeze(1)) # Adapted to the input format of RobertaClassificationHead
        # Alternatively directly: logits = self.classifier.out_proj(self.classifier.dense(pooled_output))

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {'loss': loss, 'logits': logits}

# ==========================================
# 2. Data Processing (kept identical)
# ==========================================
class MetaphorSentimentDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        print(f"Loading data: {csv_path}")
        self.data = pd.read_csv(csv_path)
        # Simple column name check and correction
        if 'sentence' in self.data.columns: self.data.rename(columns={'sentence': 'review'}, inplace=True)
        if 'sentiment' in self.data.columns: self.data.rename(columns={'sentiment': 'label'}, inplace=True)

        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def align_labels(self, tokenized_inputs, word_labels):
        word_ids = tokenized_inputs.word_ids()
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(0)
            else:
                try:
                    label = word_labels[word_idx]
                except IndexError:
                    label = 0
                label_ids.append(label)
        return label_ids

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row['review'])
        label = int(row['label'])

        metaphor_labels_str = row['metaphor_labels']
        if isinstance(metaphor_labels_str, str):
            try:
                word_labels = ast.literal_eval(metaphor_labels_str)
            except:
                word_labels = [0] * len(text.split())
        else:
            word_labels = metaphor_labels_str

        # Handle length mismatch
        words = text.split()
        min_len = min(len(word_labels), len(words))
        words = words[:min_len]
        word_labels = word_labels[:min_len]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()

        aligned_metaphor_labels = self.align_labels(encoding, word_labels)
        if len(aligned_metaphor_labels) > self.max_len:
            aligned_metaphor_labels = aligned_metaphor_labels[:self.max_len]
        else:
            aligned_metaphor_labels += [0] * (self.max_len - len(aligned_metaphor_labels))

        metaphor_features = torch.tensor(aligned_metaphor_labels, dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'metaphor_features': metaphor_features,
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 3. Evaluation and Training Functions
# ==========================================
def evaluate_detailed(model, dataloader, device):
    model.eval()
    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            metaphor_features = batch['metaphor_features'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                metaphor_features=metaphor_features
            )

            logits = outputs['logits']
            batch_preds = torch.argmax(logits, dim=1).cpu().tolist()
            preds.extend(batch_preds)
            true_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(true_labels, preds)
    p = precision_score(true_labels, preds, pos_label=1)
    r = recall_score(true_labels, preds, pos_label=1)
    f1 = f1_score(true_labels, preds, pos_label=1)

    return acc, p, r, f1

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    progress = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        metaphor_features = batch['metaphor_features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, metaphor_features=metaphor_features, labels=labels)
        loss = outputs['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix({'loss': loss.item()})

    return total_loss / len(dataloader)

# ==========================================
# 4. Main Pipeline
# ==========================================
def main():
    # Configuration
    TRAIN_FILE = '/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book/Labeled/book_metaphor_train.csv'
    TEST_FILE = '/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book/Labeled/book_metaphor_test.csv'
    MODEL_NAME = 'roberta-large'
    BATCH_SIZE = 32
    EPOCHS = 4
    LR = 1e-5

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, add_prefix_space=True)

    print("\n=== Experiment: Direct Embedding of Metaphor Features (No Attention) ===")

    # Load datasets
    try:
        train_ds = MetaphorSentimentDataset(TRAIN_FILE, tokenizer)
        test_ds = MetaphorSentimentDataset(TEST_FILE, tokenizer)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    except Exception as e:
        print(f"Data Load Error: {e}")
        return

    # Initialize Model (using the NoAttention version)
    config = RobertaConfig.from_pretrained(MODEL_NAME, num_labels=2)
    model = RobertaMetaphorNoAttention.from_pretrained(MODEL_NAME, config=config)
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*len(train_loader)*EPOCHS), num_training_steps=len(train_loader)*EPOCHS)

    best_f1 = 0
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        avg_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"  Train Loss: {avg_loss:.4f}")

        acc, prec, rec, f1 = evaluate_detailed(model, test_loader, device)
        print(f"  Valid Metrics - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            print(f"  >> New Best No-Attn Model! (F1: {best_f1:.4f})")
            # torch.save(model.state_dict(), 'no_attn_model.bin')

if __name__ == "__main__":
    main()